# Memory Usage Benchmark Development and Final Experiment

This notebook documents the design, implementation, execution and quality assurance of the memory-usage benchmark used to compare Node.js, Bun and Deno.

The benchmark evaluates memory consumption under two controlled conditions:

1. an idle runtime condition; and
2. a controlled memory-allocation condition.

Statistical analysis, visualisation and interpretation are deferred until data collection and quality assurance have been completed for all dependent variables in the study.

## 1. Purpose of the Memory Benchmark

The memory benchmark measures the operating-system-observed memory footprint of Node.js, Bun and Deno under equivalent workloads.

Two workload conditions were used.

The `idle` condition measures the runtime after the benchmark program has started and reached a stable readiness state without deliberately allocating a large application payload.

The `allocated` condition measures the runtime while retaining a controlled 100 MiB binary memory allocation.

Using both conditions allows the study to distinguish between the baseline runtime footprint and memory consumption under an equivalent application-level allocation.

## 2. Memory Workloads

The same JavaScript workload implementation was executed by Node.js, Bun and Deno.

### Idle workload

The idle workload:

1. started the JavaScript runtime;
2. performed minimal benchmark initialisation;
3. emitted a structured `READY` signal; and
4. remained alive while the external measurement tool sampled memory usage.

No large application allocation was created.

### Allocated workload

The allocated workload:

1. created a `Uint8Array` of 100 MiB;
2. filled the complete array with a fixed byte value;
3. retained a live reference to the array;
4. emitted the `READY` signal; and
5. remained alive while memory usage was sampled.

The allocation size was:

`104,857,600 bytes`

Filling the complete array ensured that the memory was actively accessed rather than merely reserving an address range. The reference was retained for the duration of the measurement so that the allocation could not be reclaimed while sampling was taking place.

## 3. Measurement Tool and Justification

Memory usage was measured externally using a Python automation runner and the `psutil` library.

The benchmark did not use runtime-specific JavaScript memory APIs as the primary measurement source. Node.js, Bun and Deno expose different runtime and engine-level memory interfaces, and directly comparing such APIs could result in different definitions of memory being measured.

Instead, the same operating-system-level measurement mechanism was applied to all three runtimes.

The primary memory quantity recorded was Resident Set Size (RSS), expressed in bytes and converted to mebibytes (MiB). Under the Windows experimental environment, RSS represents the process working set and therefore provides an externally observed measure of physical memory associated with the running process.

If a runtime created descendant processes, the runner recursively identified those processes and added their RSS values to the parent runtime's RSS. This ensured that memory belonging to runtime-related child processes was not silently excluded from the observation.

The use of an external measurement mechanism improved comparability because Node.js, Bun and Deno were observed using the same measurement interface and operating-system definition.

## 4. Measurement Procedure

The Python runner started a new runtime process for every observation and began monitoring its memory usage.

Memory was sampled every 20 milliseconds.

The JavaScript workload emitted a structured `READY` signal after the workload had reached the state intended for measurement. For the allocated condition, this occurred only after the 100 MiB allocation had been created, filled and retained.

After the readiness signal was received, memory sampling continued for two seconds.

The runner retained the complete sequence of sampled RSS values for each observation and calculated summary measures including:

- RSS at readiness;
- median post-readiness RSS;
- mean post-readiness RSS;
- peak post-readiness RSS; and
- peak RSS observed during the complete process lifetime.

Median post-readiness RSS was selected as the primary steady-state memory measure because it represents the typical memory footprint across the sampling window and is less sensitive than a single sample to short-lived fluctuations.

Peak post-readiness RSS was retained as a secondary measure of the maximum observed memory requirement while the runtime remained in the controlled workload state.

## 5. Final Experimental Protocol

| Experimental property | Final configuration |
|---|---|
| Runtimes | Node.js, Bun and Deno |
| Workloads | Idle and allocated |
| Allocation size | 100 MiB |
| Repetitions | 10 per runtime-workload combination |
| Sessions | 2 |
| Repetitions per session | 5 |
| Runtime-workload groups | 6 |
| Observations per session | 30 |
| Total observations | 60 |
| Sampling interval | 20 ms |
| Post-readiness sampling window | 2 seconds |
| Primary measure | Median post-readiness RSS |
| Secondary measure | Peak post-readiness RSS |
| Unit | MiB |
| Process reuse | No |
| Execution order | Randomised within each repetition |
| Cool-down | 2 seconds |

The six runtime-workload combinations were randomised independently within each repetition using a fixed random seed.

Session one contained repetitions 1–5 and session two contained repetitions 6–10.

A fresh runtime process was created for every observation.

### 5.1 Repetition Count and Memory Metric Justification

Twenty independent repetitions were collected for each runtime-workload combination. Repeated executions are important in software benchmarking because performance measurements can vary between otherwise equivalent runs. Laaber et al. (2021) investigated benchmark measurements using 5, 10, 20 and 30 iterations, demonstrating the importance of repeated measurements when assessing benchmark stability. Twenty repetitions were therefore selected as a practical balance between measurement reliability and experimental execution time.

With three runtimes and two memory workloads, this produced:

- 3 runtimes;
- 2 workload conditions;
- 20 independent repetitions per condition; and
- **120 final memory observations**.

The multiple RSS samples collected during an individual process execution were not treated as separate independent observations. Instead, each newly created runtime process represented one experimental observation.

Resident Set Size (RSS) was used as the operating-system-level memory metric. RSS represents the resident-memory footprint associated with a running process and provides a common external measure that can be applied consistently to Node.js, Bun and Deno. Pusukuri (2014), in research on working-set modelling for multithreaded programs, used RSS as an important predictor when characterising application working-set behaviour, supporting its relevance as an application memory characteristic.

Using RSS also avoids relying on runtime-specific JavaScript heap-reporting APIs, which may expose different internal definitions across the three runtimes.

**References**

Laaber, C. et al. (2021). *Predicting unstable software benchmarks using static source code features*. Empirical Software Engineering, 26. https://doi.org/10.1007/s10664-021-09996-y.

Pusukuri, K.K. (2014). *Working Set Model for Multithreaded Programs*. 2014 Conference on Timely Results in Operating Systems (TRIOS '14), USENIX Association.

In [2]:
library(dplyr)

candidate_manifest_paths <- c(
  "data/raw/memory/memory_final_manifest.csv",
  "../data/raw/memory/memory_final_manifest.csv"
)

available_manifest_paths <- candidate_manifest_paths[
  file.exists(candidate_manifest_paths)
]

if (length(available_manifest_paths) == 0) {
  stop("The final memory manifest could not be found.")
}

memory_manifest_path <- normalizePath(
  available_manifest_paths[1],
  winslash = "/",
  mustWork = TRUE
)

memory_final <- read.csv(
  memory_manifest_path,
  stringsAsFactors = FALSE
)

memory_final <- memory_final |>
  mutate(
    session_id = as.integer(session_id),
    sequence = as.integer(sequence),
    repetition = as.integer(repetition),
    allocation_bytes = as.numeric(allocation_bytes),
    sample_interval_ms = as.numeric(sample_interval_ms),
    post_ready_window_ms = as.numeric(post_ready_window_ms),
    pre_ready_samples = as.integer(pre_ready_samples),
    post_ready_samples = as.integer(post_ready_samples),
    rss_at_ready_mib = as.numeric(rss_at_ready_mib),
    median_post_ready_rss_mib = as.numeric(
      median_post_ready_rss_mib
    ),
    mean_post_ready_rss_mib = as.numeric(
      mean_post_ready_rss_mib
    ),
    peak_post_ready_rss_mib = as.numeric(
      peak_post_ready_rss_mib
    ),
    peak_observed_rss_mib = as.numeric(
      peak_observed_rss_mib
    )
  )

cat("Manifest path:\n")
cat(memory_manifest_path, "\n\n")

cat("Rows:", nrow(memory_final), "\n")
cat("Columns:", ncol(memory_final), "\n")


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




Manifest path:
D:/FOLO_PROJECTS/MASTERS/javascript-runtime-performance-study/data/raw/memory/memory_final_manifest.csv 

Rows: 120 
Columns: 31 


In [3]:
#check each workload group

memory_group_counts <- memory_final |>
  count(
    runtime,
    workload,
    name = "observations"
  ) |>
  arrange(
    workload,
    runtime
  )

memory_group_counts

runtime,workload,observations
<chr>,<chr>,<int>
bun,allocated,20
deno,allocated,20
node,allocated,20
bun,idle,20
deno,idle,20
node,idle,20


In [4]:
#check the 2 sessions

memory_session_counts <- memory_final |>
  count(
    session_id,
    runtime,
    workload,
    name = "observations"
  ) |>
  arrange(
    session_id,
    workload,
    runtime
  )

memory_session_counts

session_id,runtime,workload,observations
<int>,<chr>,<chr>,<int>
1,bun,allocated,10
1,deno,allocated,10
1,node,allocated,10
1,bun,idle,10
1,deno,idle,10
1,node,idle,10
2,bun,allocated,10
2,deno,allocated,10
2,node,allocated,10


In [5]:
#check for duplicates
memory_duplicate_check <- memory_final |>
  count(
    runtime,
    workload,
    repetition,
    name = "occurrences"
  ) |>
  filter(
    occurrences != 1
  )

memory_duplicate_check

runtime,workload,repetition,occurrences
<chr>,<chr>,<int>,<int>


In [6]:
#check missing configurations
expected_memory_design <- expand.grid(
  runtime = c(
    "node",
    "bun",
    "deno"
  ),
  workload = c(
    "idle",
    "allocated"
  ),
  repetition = 1:20,
  stringsAsFactors = FALSE
)

missing_memory_observations <- expected_memory_design |>
  anti_join(
    memory_final |>
      select(
        runtime,
        workload,
        repetition
      ),
    by = c(
      "runtime",
      "workload",
      "repetition"
    )
  )

missing_memory_observations

runtime,workload,repetition
<chr>,<chr>,<int>


In [7]:
memory_allocation_check <- memory_final |>
  mutate(
    expected_allocation = ifelse(
      workload == "allocated",
      104857600,
      0
    )
  ) |>
  filter(
    allocation_bytes != expected_allocation
  )

memory_allocation_check

Warning message in cbind(parts$left, chars$ellip_h, parts$right, deparse.level = 0L):
"number of rows of result is not a multiple of vector length (arg 2)"
Warning message in cbind(parts$left, chars$ellip_h, parts$right, deparse.level = 0L):
"number of rows of result is not a multiple of vector length (arg 2)"
Warning message in cbind(parts$left, chars$ellip_h, parts$right, deparse.level = 0L):
"number of rows of result is not a multiple of vector length (arg 2)"
Warning message in cbind(parts$left, chars$ellip_h, parts$right, deparse.level = 0L):
"number of rows of result is not a multiple of vector length (arg 2)"


run_id,session_id,sequence,repetition,runtime,workload,shuffle_seed,started_at,ready_at,finished_at,⋯,peak_post_ready_rss_bytes,peak_post_ready_rss_mib,peak_observed_rss_bytes,peak_observed_rss_mib,ready_payload,sample_file,stdout_log,stderr_log,error_message,expected_allocation
<chr>,<int>,<int>,<int>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,⋯,<int>,<dbl>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>,<lgl>,<dbl>


In [8]:
#check sampling config

memory_sampling_check <- memory_final |>
  summarise(
    incorrect_sample_intervals = sum(
      sample_interval_ms != 20,
      na.rm = TRUE
    ),

    incorrect_sampling_windows = sum(
      post_ready_window_ms != 2000,
      na.rm = TRUE
    ),

    insufficient_post_ready_samples = sum(
      post_ready_samples < 10,
      na.rm = TRUE
    )
  )

memory_sampling_check

incorrect_sample_intervals,incorrect_sampling_windows,insufficient_post_ready_samples
<int>,<int>,<int>
0,0,0


In [9]:
# full audit

memory_audit_summary <- data.frame(
  check = c(
    "Total observations",
    "Unique run identifiers",
    "Runtime-workload groups",
    "Groups with 20 observations",
    "Session-runtime-workload groups",
    "Session groups with 10 observations",
    "Duplicate configurations",
    "Missing configurations",
    "Failed observations",
    "Incorrect allocation sizes",
    "Missing median RSS values",
    "Non-positive median RSS values",
    "Missing peak RSS values",
    "Non-positive peak RSS values",
    "Median RSS greater than post-ready peak",
    "Post-ready peak greater than overall peak",
    "Incorrect sampling intervals",
    "Incorrect post-ready windows",
    "Insufficient post-ready samples"
  ),

  result = c(
    nrow(memory_final),

    n_distinct(
      memory_final$run_id
    ),

    nrow(
      memory_group_counts
    ),

    sum(
      memory_group_counts$observations == 20
    ),

    nrow(
      memory_session_counts
    ),

    sum(
      memory_session_counts$observations == 10
    ),

    nrow(
      memory_duplicate_check
    ),

    nrow(
      missing_memory_observations
    ),

    sum(
      memory_final$status != "success"
    ),

    nrow(
      memory_allocation_check
    ),

    sum(
      is.na(
        memory_final$median_post_ready_rss_mib
      )
    ),

    sum(
      memory_final$median_post_ready_rss_mib <= 0,
      na.rm = TRUE
    ),

    sum(
      is.na(
        memory_final$peak_post_ready_rss_mib
      )
    ),

    sum(
      memory_final$peak_post_ready_rss_mib <= 0,
      na.rm = TRUE
    ),

    sum(
      memory_final$median_post_ready_rss_mib >
        memory_final$peak_post_ready_rss_mib,
      na.rm = TRUE
    ),

    sum(
      memory_final$peak_post_ready_rss_mib >
        memory_final$peak_observed_rss_mib,
      na.rm = TRUE
    ),

    memory_sampling_check$incorrect_sample_intervals,

    memory_sampling_check$incorrect_sampling_windows,

    memory_sampling_check$insufficient_post_ready_samples
  ),

  expected = c(
    120,
    120,
    6,
    6,
    12,
    12,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
  )
) |>
  mutate(
    passed = result == expected
  )

memory_audit_summary

check,result,expected,passed
<chr>,<int>,<dbl>,<lgl>
Total observations,120,120,TRUE
Unique run identifiers,120,120,TRUE
Runtime-workload groups,6,6,TRUE
Groups with 20 observations,6,6,TRUE
Session-runtime-workload groups,12,12,TRUE
Session groups with 10 observations,12,12,TRUE
Duplicate configurations,0,0,TRUE
Missing configurations,0,0,TRUE
Failed observations,0,0,TRUE


In [10]:
#save the processed data
project_root <- if (
  dir.exists("data")
) {
  "."
} else {
  ".."
}

memory_processed_directory <- file.path(
  project_root,
  "data",
  "processed",
  "memory"
)

memory_table_directory <- file.path(
  project_root,
  "results",
  "tables",
  "memory"
)

dir.create(
  memory_processed_directory,
  recursive = TRUE,
  showWarnings = FALSE
)

dir.create(
  memory_table_directory,
  recursive = TRUE,
  showWarnings = FALSE
)

memory_processed_path <- file.path(
  memory_processed_directory,
  "memory_final.csv"
)

write.csv(
  memory_final,
  memory_processed_path,
  row.names = FALSE
)

write.csv(
  memory_audit_summary,
  file.path(
    memory_table_directory,
    "memory_final_audit_summary.csv"
  ),
  row.names = FALSE
)

cat(
  "Processed dataset saved to:\n",
  normalizePath(
    memory_processed_path,
    winslash = "/",
    mustWork = TRUE
  )
)

Processed dataset saved to:
 D:/FOLO_PROJECTS/MASTERS/javascript-runtime-performance-study/data/processed/memory/memory_final.csv

## 6. Final Data Collection and Quality Assurance

The final memory experiment produced 120 independent observations. The experimental design consisted of:

- three JavaScript runtimes;
- two controlled memory workloads: idle and allocated; and
- twenty independent repetitions per runtime-workload combination.

The experiment was divided into two balanced sessions. Each session contained ten repetitions of all six runtime-workload combinations, producing 60 observations per session.

Within every repetition, the runtime-workload execution order was randomised using a fixed and recorded random seed.

For the allocated workload, each process created, accessed and retained a 100 MiB (`104,857,600` byte) `Uint8Array` before signalling readiness. The idle workload created no corresponding deliberate allocation.

RSS was sampled every 20 milliseconds for two seconds after the workload emitted its `READY` signal. The complete sampling trace was retained for every observation.

A post-collection quality audit confirmed that:

- all 120 planned observations were present;
- all run identifiers were unique;
- all six runtime-workload groups contained exactly twenty observations;
- each runtime-workload group contained ten observations in each session;
- no configurations were duplicated or missing;
- all observations completed successfully;
- the controlled allocation sizes matched the intended workload;
- all primary and secondary RSS measurements were present and greater than zero;
- the recorded RSS summaries were internally consistent;
- the sampling interval and post-readiness measurement window matched the frozen protocol; and
- each observation contained sufficient post-readiness RSS samples.

The audited dataset was saved as:

`data/processed/memory/memory_final.csv`

The raw observation files, complete RSS sampling traces, execution plans, logs, program hashes and frozen experimental metadata were retained.

## 7. Deferred Analysis

Descriptive statistics, visualisations, inferential statistical testing and interpretation of the memory results were deferred until data collection and quality assurance had been completed for all dependent variables.